In [1]:
"""Quick check: Genesis radiative heating contribution."""

import numpy as np
from reentrykit.trajectory import InitialState, Vehicle, simulate
from reentrykit.planet import EARTH_NON_ROTATING
from reentrykit.aerothermal import heating_history

# Genesis vehicle (from notebook 05)
GENESIS_MASS = 225.0
GENESIS_DIAMETER = 1.52
GENESIS_AREA = np.pi * (GENESIS_DIAMETER / 2.0) ** 2
GENESIS_NOSE_RADIUS = 0.664
GENESIS_CD = 1.05

vehicle = Vehicle.from_mass_area_cd(
    mass=GENESIS_MASS, reference_area=GENESIS_AREA,
    drag_coefficient=GENESIS_CD, lift_to_drag_ratio=0.0, bank_angle=0.0,
    nose_radius=GENESIS_NOSE_RADIUS,
)
state = InitialState(
    altitude=125_000.0, velocity=11_000.0,
    flight_path_angle=np.deg2rad(-8.0), heading=np.deg2rad(12.0),
    latitude=np.deg2rad(39.0), longitude=np.deg2rad(-125.0),
)
trajectory = simulate(vehicle, state, planet=EARTH_NON_ROTATING,
                       max_time=500.0, dt_output=0.1)
heat = heating_history(trajectory, nose_radius=GENESIS_NOSE_RADIUS)

print(f"Genesis heating with Tauber-Sutton radiative:")
print(f"  Peak convective:  {heat.peak_convective_flux/1e6:.2f} MW/m²")
print(f"  Peak radiative:   {heat.peak_radiative_flux/1e6:.2f} MW/m²")
print(f"  Peak total:       {heat.peak_total_flux/1e6:.2f} MW/m²")
print(f"  Total heat load:  {heat.total_integrated_load/1e6:.1f} MJ/m²")
print(f"  Radiative fraction at peak: "
      f"{heat.peak_radiative_flux/heat.peak_total_flux*100:.1f}%")

# Also: when does radiation actually fire?
i_max_rad = int(heat.radiative_flux.argmax())
print(f"\nMax radiative flux occurs at:")
print(f"  Time: {heat.time[i_max_rad]:.1f} s")
print(f"  Altitude: {trajectory.altitude[i_max_rad]/1000:.1f} km")
print(f"  Velocity: {trajectory.velocity[i_max_rad]:.0f} m/s")
print(f"  Density:  {trajectory.density[i_max_rad]:.2e} kg/m³")

Genesis heating with Tauber-Sutton radiative:
  Peak convective:  4.02 MW/m²
  Peak radiative:   0.93 MW/m²
  Peak total:       4.71 MW/m²
  Total heat load:  140.6 MJ/m²
  Radiative fraction at peak: 19.8%

Max radiative flux occurs at:
  Time: 48.5 s
  Altitude: 61.7 km
  Velocity: 10339 m/s
  Density:  2.33e-04 kg/m³
